# 02b - PCA, NMF and UMAP

Three complementary views of the same pixels: PCA for variance structure,
NMF for additive chemical endmembers, UMAP for non-linear neighbourhood
structure.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [1]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
# Show paths relative to the project root so notebooks stay portable.
print("Project  :", ROOT.name)
print("Raw data :", config.raw_dir.relative_to(ROOT) if ROOT in config.raw_dir.parents else config.raw_dir)
print("Outputs  :", config.processed_dir.relative_to(ROOT) if ROOT in config.processed_dir.parents else config.processed_dir)

Project  : Fossil Fly
Raw data : data\raw
Outputs  : data\processed


In [2]:
from src.features import decomposition as decomp
from src.features.selection import build_cube, select_analysis_channels
from src.preprocessing.stack_io import load_clean_stack

images, metadata = load_clean_stack(config.processed_dir)
selection = select_analysis_channels(images, metadata, config)
cube = build_cube(images, selection)

prepared = decomp.prepare_pixels(cube)
print(f"{prepared.n_foreground:,} foreground pixels of {len(prepared.fg_mask):,}")

48,294 foreground pixels of 409,600


## PCA

Fitted on standardized intensities so no single high-count channel dominates.

In [3]:
from src.viz import decomposition_plots

pca = decomp.run_pca(prepared, random_state=42)
print("explained variance (%):", (pca["explained_variance"] * 100).round(2))

decomposition_plots.plot_pca_scree(
    pca, selection.labels, config.figure_path("02b_pca_scree.png")
)
decomposition_plots.plot_pca_component_maps(
    pca, prepared.fg_mask, config.figure_path("02b_pca_component_maps.png")
)
print("figures written")

explained variance (%): [36.98 23.66 19.77 16.45  3.13]


figures written


## NMF

Non-negativity makes the components physically interpretable: each pixel is a
sum of endmember contributions, never a cancellation.

The rank is chosen from the elbow of the reconstruction-error curve rather
than fixed, so a new dataset gets its own appropriate rank.

In [4]:
sweep = decomp.run_nmf_sweep(prepared, sweep=config.get("decomposition.nmf.sweep"))
for k in sorted(sweep):
    print(f"  k={k}: reconstruction error {sweep[k]['error']:,.1f}")

chosen_k = decomp.select_nmf_rank(sweep)
print(f"\nelbow -> k = {chosen_k}")

  k=2: reconstruction error 248,987.5
  k=3: reconstruction error 145,738.2
  k=4: reconstruction error 144,089.8
  k=5: reconstruction error 117,354.3

elbow -> k = 3


In [5]:
nmf = decomp.run_nmf(prepared, n_components=chosen_k, sweep_results=sweep)

decomposition_plots.plot_nmf_model_selection(
    sweep, chosen_k, config.figure_path("02b_nmf_model_selection.png")
)
decomposition_plots.plot_nmf_endmembers(
    nmf, selection.labels, config.figure_path("02b_nmf_endmembers.png")
)
decomposition_plots.plot_nmf_abundance_maps(
    nmf, config.figure_path("02b_nmf_abundance_maps.png")
)
decomposition_plots.plot_nmf_rgb_composite(
    nmf, config.figure_path("02b_nmf_rgb_composite.png")
)
print("figures written")

figures written


In [6]:
import pandas as pd

# Endmember spectra: which masses define each component.
pd.DataFrame(
    nmf["H"],
    index=[f"EM{i + 1}" for i in range(chosen_k)],
    columns=selection.labels,
).round(2)

,m/z 62.96 (Neg),m/z 78.95 (Neg),m/z 96.94 (Neg),m/z 103.91 (Neg),m/z 123.95 (Neg)
EM1,0.00,3.56,8.69,0.00,1349.28
EM2,556.34,886.34,12.94,0.00,0.00
EM3,0.00,11.89,16.24,615.89,78.12


## UMAP

Only foreground pixels are embedded; background is a single degenerate point
that would otherwise dominate the neighbourhood graph.

In [7]:
umap_result = None
if config.get("decomposition.umap.enabled", True):
    umap_result = decomp.run_umap(prepared)
    decomposition_plots.plot_umap_scatter(
        umap_result, prepared, nmf, config.figure_path("02b_umap_scatter.png")
    )
    decomposition_plots.plot_umap_spatial_maps(
        umap_result, config.figure_path("02b_umap_spatial_maps.png")
    )
    print(f"embedded {umap_result['embedding'].shape[0]:,} pixels")

embedded 48,294 pixels


## Save for the next stage

In [8]:
written = decomp.save_decomposition(
    config.processed_dir, prepared, pca, nmf, umap_result,
    selection.labels, selection.keys,
)
for name, path in written.items():
    print(f"{name:18s} -> {path.name}")

pca                -> 02b_pca_components.npz


nmf                -> 02b_nmf_components.npz
umap               -> 02b_umap_embedding.npz
foreground_mask    -> 02b_foreground_mask.npy


Equivalent CLI command:

```bash
python scripts/run_pipeline.py --stage decompose
```